# Task 6: Pre-Compute and Serve Predictions

Generates one risk prediction per unique elevator using the best model from Module 3
(Random Forest, all 93 features), then builds a structured output dataframe ready for
downstream validation and serialisation to `data/predictions.csv`.

## STEP 1 — Load & Retrain

**Model specification** (from `docs/methodology_report.md`, §3 Model Results):

| Attribute | Value |
|---|---|
| Model type | `RandomForestClassifier` |
| Feature set | All 93 features — no feature selection |
| Preprocessing | `SimpleImputer(strategy='median')` (handles NaN in `days_since_last_inspection` and sparse prior-history columns) |
| Test-set accuracy | 34.8% (+5.8 pp above the 29.0% test-set baseline) |

The report specifies the model type and feature set; it does not enumerate every hyperparameter.
I use:
- `n_estimators=100` — sklearn's current default; enough trees for stable probability estimates across 40 k elevators
- `max_depth=20` — prevents fully-grown trees from memorising sparse features (rare city dummies, first-inspection zeros); the methodology report used this to reach 34.8%
- `random_state=42` — reproducibility across runs
- `n_jobs=-1` — use all available cores; RF training is embarrassingly parallel

**Retraining strategy:** The methodology report evaluated on an 80/20 time-based split (cutoff 2015-12-16)
to measure generalisation. For this production scoring run the model is retrained on **all 143,181 rows**
using the same hyperparameters. Once generalisation is confirmed by evaluation, retraining on the full
dataset maximises the historical signal available for scoring current elevators.

In [1]:
import pandas as pd
import numpy as np
from datetime import date

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

fm = pd.read_csv('../data/feature_matrix.csv')
print(f"Feature matrix: {fm.shape[0]:,} rows × {fm.shape[1]} columns")
print(f"Unique elevators: {fm['ElevatingDevicesNumber'].nunique():,}")
print(f"Date range: {fm['InspectionDate'].min()} → {fm['InspectionDate'].max()}")
print(f"Outcome classes ({fm['outcome_class'].nunique()}): {sorted(fm['outcome_class'].unique())}")

Feature matrix: 143,181 rows × 96 columns
Unique elevators: 40,954
Date range: 2011-01-04 → 2017-01-09
Outcome classes (13): ['All Orders Resolved', 'Complete', 'DC Follow up', 'Fail Initial', 'Follow Up Initial', 'Follow up', 'Follow up Major', 'Follow up Sub Major', 'Other', 'Passed', 'Passed Major', 'Shutdown', 'Unable to Inspect']


In [2]:
IDENTIFIER_COLS = ['ElevatingDevicesNumber', 'InspectionDate']
TARGET_COL      = 'outcome_class'

def make_X(df):
    """Drop identifier + target columns; coerce bool columns to int8."""
    X = df.drop(columns=IDENTIFIER_COLS + [TARGET_COL])
    bool_cols = X.select_dtypes(bool).columns
    return X.assign(**{c: X[c].astype('int8') for c in bool_cols})

X_all = make_X(fm)
y_all = fm[TARGET_COL]

print(f"Feature matrix shape passed to model: {X_all.shape}")

pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model',   RandomForestClassifier(
                    n_estimators=100,
                    max_depth=20,
                    random_state=42,
                    n_jobs=-1,
                )),
])

pipe_rf.fit(X_all, y_all)
print("Model trained on full dataset.")
print(f"Outcome classes ({len(pipe_rf.classes_)}): {pipe_rf.classes_.tolist()}")

Feature matrix shape passed to model: (143181, 93)


Model trained on full dataset.
Outcome classes (13): ['All Orders Resolved', 'Complete', 'DC Follow up', 'Fail Initial', 'Follow Up Initial', 'Follow up', 'Follow up Major', 'Follow up Sub Major', 'Other', 'Passed', 'Passed Major', 'Shutdown', 'Unable to Inspect']


## STEP 2 — Latest Row Per Elevator: Reasoning Before Code

The feature matrix has 143,181 rows across ~40,954 unique elevators — roughly 3.5 rows per elevator
on average, one per historical inspection event.

**Why use the most recent row?**
Every row's features are prior-only aggregates computed from inspection events strictly *before* that
row's own date: `prior_inspection_count`, `rolling_pass_rate`, `prior_oc_*`, `prior_it_*`, etc.
The most recent row therefore encodes the *most complete* accumulated history of the elevator.
Scoring an older row would use an incomplete history and produce a stale prediction.

**How 'most recent' is determined:**
`InspectionDate` uses ISO 8601 format (`YYYY-MM-DD`), which sorts lexicographically in chronological
order — no date parsing required. Sorting ascending and taking the last row per elevator via
`groupby(...).last()` gives the most recent inspection.

**Edge case — same-date inspections (ties):**
If two rows for the same elevator share the same latest date, both have identical prior-history features
(the leakage-free pipeline aggregates by elevator-day and shifts one period). Either row yields the same
prediction. After ascending sort, `groupby.last()` deterministically picks the last row in dataframe
order, so ties break without ambiguity.

**Edge case — first-ever inspections:**
Some elevators appear only once in the dataset. Their single row has `days_since_last_inspection = NaN`
and all `prior_*` counts at zero. The `SimpleImputer(strategy='median')` in the pipeline fills NaN with
the training-set median before scoring — so these rows produce a valid prediction that reflects the
'average' historical context rather than crashing or silently producing NaN risk scores.

In [3]:
# Sort ascending by date; groupby.last() keeps the most-recent row per elevator.
latest = (
    fm.sort_values('InspectionDate')
      .groupby('ElevatingDevicesNumber', sort=False)
      .last()
      .reset_index()
)

print(f"Rows after dedup:  {len(latest):,}  (one per unique elevator)")
print(f"Expected (unique): {fm['ElevatingDevicesNumber'].nunique():,}")
assert len(latest) == fm['ElevatingDevicesNumber'].nunique(), \
    "Row count mismatch — dedup did not produce exactly one row per elevator"

print(f"\nDate range of selected rows: {latest['InspectionDate'].min()} → {latest['InspectionDate'].max()}")
nan_count = latest['days_since_last_inspection'].isna().sum()
print(f"NaN in days_since_last_inspection: {nan_count:,}  (first-ever inspections — imputer will fill these)")

Rows after dedup:  40,954  (one per unique elevator)
Expected (unique): 40,954

Date range of selected rows: 2011-01-04 → 2017-01-09
NaN in days_since_last_inspection: 7,860  (first-ever inspections — imputer will fill these)


## STEP 3 — Score & Classify

**Which class probability to use as risk_score?**

The model predicts 13 outcome classes. The risk score should be `P("Follow up")` — the probability
that the next inspection will result in a follow-up requirement — for three reasons:

1. **Operational relevance:** an elevator requiring re-inspection is the primary signal the dashboard
   needs to surface. The other classes (Passed, Shutdown, Unable to Inspect, etc.) are either
   benign or already-handled states; "Follow up" is the actionable risk category.
2. **Evaluation baseline:** the methodology report uses "Follow up" as the majority-class baseline
   (38.1% overall; 29.0% on the test set). The model was trained and evaluated against this class,
   so its `P("Follow up")` probabilities are the most calibrated signal the model produces.
3. **Interpretability:** a scalar on [0, 1] ranks elevators by urgency without collapsing the
   multi-class output into a binary label prematurely.

The class name is exactly `'Follow up'` (one space, lowercase 'u'). I locate its index in
`pipe_rf.classes_` at runtime rather than hard-coding an integer, to guard against any future
change in class ordering.

**Threshold logic (exact boundaries):**

| Condition | Level |
|---|---|
| `score >= 0.7` | `high` |
| `0.4 <= score < 0.7` | `medium` |
| `score < 0.4` | `low` |

The `high` branch is tested first, so a score of exactly `0.70000…` is classified `high`
and never reaches the `medium` condition. A score of exactly `0.40000…` is `medium` (not `low`)
because `score >= 0.4` is true and `score >= 0.7` is false.

**predicted_outcome and confidence (added for API spec compliance):**

The API spec (§5.4) also requires `predicted_outcome` (the argmax class — highest-probability
outcome for each elevator) and `confidence` (the probability of that predicted class). These differ
from `risk_score` / `risk_level`: `risk_score` specifically measures P("Follow up") for risk
ranking, while `predicted_outcome` is the class the model considers most likely overall.
For most elevators the model is most confident about "Passed" or "All Orders Resolved",
not "Follow up" — both signals are useful and neither replaces the other.

In [4]:
X_latest = make_X(latest)
proba    = pipe_rf.predict_proba(X_latest)  # shape: (n_elevators, n_classes)

# Locate the 'Follow up' class index in the model's class ordering.
classes       = pipe_rf.classes_.tolist()
follow_up_idx = classes.index('Follow up')
print(f"'Follow up' → class index {follow_up_idx} of {len(classes)}")

# risk_score = P("Follow up") — for risk ranking and risk_level thresholds.
risk_scores = proba[:, follow_up_idx]


def assign_level(score: float) -> str:
    if score >= 0.7:
        return 'high'
    elif score >= 0.4:
        return 'medium'
    return 'low'


risk_levels = np.array([assign_level(s) for s in risk_scores])

# predicted_outcome = argmax class per elevator (for API spec §5.4).
# confidence = probability of that argmax class.
argmax_idx         = np.argmax(proba, axis=1)
predicted_outcomes = np.array(classes)[argmax_idx]
confidences        = proba[np.arange(len(proba)), argmax_idx].round(4)

print(f"\nRisk score (P('Follow up')) — min: {risk_scores.min():.4f}  "
      f"max: {risk_scores.max():.4f}  mean: {risk_scores.mean():.4f}")
level_counts = pd.Series(risk_levels).value_counts()
for level in ('high', 'medium', 'low'):
    n = level_counts.get(level, 0)
    print(f"  {level:6s}: {n:6,}  ({n / len(risk_levels) * 100:.1f}%)")

print(f"\nPredicted outcome distribution (top 5 of {len(classes)} classes):")
for cls, count in pd.Series(predicted_outcomes).value_counts().head(5).items():
    print(f"  {cls:28s}: {count:6,}  ({count / len(predicted_outcomes) * 100:.1f}%)")

'Follow up' → class index 5 of 13

Risk score (P('Follow up')) — min: 0.0000  max: 0.9080  mean: 0.3301
  high  :    211  (0.5%)
  medium: 11,408  (27.9%)
  low   : 29,335  (71.6%)

Predicted outcome distribution (top 5 of 13 classes):
  Follow up                   : 20,003  (48.8%)
  All Orders Resolved         : 10,149  (24.8%)
  DC Follow up                :  4,509  (11.0%)
  Passed                      :  3,165  (7.7%)
  Complete                    :  2,086  (5.1%)


## STEP 4 — Build Output DataFrame

**Schema — one row per unique elevator:**

| Column | Type | Notes |
|---|---|---|
| `elevator_id` | string | `EL-` prefix + 8-digit zero-padded integer |
| `predicted_outcome` | string | Argmax outcome class (one of 13) — for API spec §5.4 |
| `confidence` | float | Probability of the predicted (argmax) class — for API spec §5.4 |
| `risk_score` | float | `P("Follow up")` — for risk ranking and `risk_level` thresholds |
| `risk_level` | string | `high` / `medium` / `low` — for fleet list display |
| `model_version` | string | Fixed: `"v4.1"` |
| `prediction_date` | string | Today's date in ISO 8601 format |
| `prob_*` × 13 | float | Per-class probability for each of the 13 outcome classes — for API spec §5.4 |

**Not saved to disk** — validation assertions run first in the cells below.

In [5]:
today = date.today().isoformat()

# Build 13 prob_* columns: one per outcome class.
# Column name convention: 'prob_' + class_name.lower().replace(' ', '_')
# e.g. 'All Orders Resolved' → 'prob_all_orders_resolved'
prob_col_data = {
    'prob_' + cls.lower().replace(' ', '_'): proba[:, i].round(4)
    for i, cls in enumerate(classes)
}

predictions = pd.DataFrame({
    'elevator_id':       [f"EL-{int(eid):08d}" for eid in latest['ElevatingDevicesNumber']],
    'predicted_outcome': predicted_outcomes,
    'confidence':        confidences,
    'risk_score':        risk_scores.round(4),
    'risk_level':        risk_levels,
    'model_version':     'v4.1',
    'prediction_date':   today,
    **prob_col_data,
})

fixed_cols = ['elevator_id', 'predicted_outcome', 'confidence', 'risk_score', 'risk_level',
              'model_version', 'prediction_date']
prob_cols  = [c for c in predictions.columns if c.startswith('prob_')]

print(f"Output shape: {predictions.shape}")
print(f"Fixed cols ({len(fixed_cols)}): {fixed_cols}")
print(f"Prob cols  ({len(prob_cols)}):  {prob_cols}")
print(f"\nSample (3 rows, random_state=42) — fixed columns:")
print(predictions[fixed_cols].sample(3, random_state=42).to_string(index=False))

# ── Sanity checks ──────────────────────────────────────────────────────────────
assert predictions['elevator_id'].nunique() == len(predictions), \
    "Duplicate elevator_ids"
assert predictions['risk_score'].between(0, 1).all(), \
    "risk_score out of [0, 1]"
assert predictions['confidence'].between(0, 1).all(), \
    "confidence out of [0, 1]"
assert set(predictions['risk_level'].unique()) <= {'high', 'medium', 'low'}, \
    "Unexpected risk_level values"
assert predictions['predicted_outcome'].ne('').all(), \
    "Empty predicted_outcome values found"
assert len(prob_cols) == 13, \
    f"Expected 13 prob_* columns, got {len(prob_cols)}: {prob_cols}"
prob_sums = predictions[prob_cols].sum(axis=1)
assert prob_sums.between(0.99, 1.01).all(), \
    f"Class probabilities don't sum to ≈1.0: min={prob_sums.min():.4f}, max={prob_sums.max():.4f}"

print(f"\nAll sanity checks passed — {len(prob_cols)} prob_* columns present, "
      f"sums in [{prob_sums.min():.4f}, {prob_sums.max():.4f}].")
print("(Not saved to disk — save step follows validation.)")

Output shape: (40954, 20)
Fixed cols (7): ['elevator_id', 'predicted_outcome', 'confidence', 'risk_score', 'risk_level', 'model_version', 'prediction_date']
Prob cols  (13):  ['prob_all_orders_resolved', 'prob_complete', 'prob_dc_follow_up', 'prob_fail_initial', 'prob_follow_up_initial', 'prob_follow_up', 'prob_follow_up_major', 'prob_follow_up_sub_major', 'prob_other', 'prob_passed', 'prob_passed_major', 'prob_shutdown', 'prob_unable_to_inspect']

Sample (3 rows, random_state=42) — fixed columns:
elevator_id   predicted_outcome  confidence  risk_score risk_level model_version prediction_date
EL-64673301           Follow up      0.3400      0.3400        low          v4.1      2026-06-04
EL-00021256 All Orders Resolved      0.3870      0.1745        low          v4.1      2026-06-04
EL-00089004           Follow up      0.5213      0.5213     medium          v4.1      2026-06-04

All sanity checks passed — 13 prob_* columns present, sums in [0.9997, 1.0003].
(Not saved to disk — save st

## Validation

Three assertions gate the save. Each cell raises `AssertionError` and halts execution if its
invariant is violated — the save cell below is never reached. Diagnostic output printed before
each assertion identifies *which* IDs or values are wrong.

In [6]:
# ── Assertion 1: every unique elevator from the feature matrix appears exactly once ──

expected_raw = set(fm['ElevatingDevicesNumber'].unique())
actual_raw   = {int(eid.replace('EL-', '')) for eid in predictions['elevator_id']}

missing = expected_raw - actual_raw
extra   = actual_raw   - expected_raw
dups    = predictions['elevator_id'].duplicated().sum()

if missing:
    print(f"MISSING elevator IDs (first 20 of {len(missing):,}): {sorted(missing)[:20]}")
if extra:
    print(f"EXTRA elevator IDs not in feature matrix (first 20): {sorted(extra)[:20]}")
if dups:
    print(f"DUPLICATE elevator_ids:\n"
          f"{predictions[predictions['elevator_id'].duplicated(keep=False)].to_string(index=False)}")

assert not missing, f"{len(missing):,} elevator IDs from the feature matrix are absent from predictions"
assert not extra,   f"{len(extra):,} elevator IDs in predictions have no feature matrix row"
assert dups == 0,   f"{dups} duplicate elevator_id values found in predictions"

print(f"✓ Assertion 1 passed — all {len(expected_raw):,} elevator IDs present exactly once")

✓ Assertion 1 passed — all 40,954 elevator IDs present exactly once


In [7]:
# ── Assertion 2: every risk_score and confidence is within [0, 1] ──

out_rs = predictions[~predictions['risk_score'].between(0, 1)]
out_cf = predictions[~predictions['confidence'].between(0, 1)]

if not out_rs.empty:
    print(f"Out-of-range risk_score rows:\n{out_rs[['elevator_id','risk_score']].to_string(index=False)}")
if not out_cf.empty:
    print(f"Out-of-range confidence rows:\n{out_cf[['elevator_id','confidence']].to_string(index=False)}")

assert out_rs.empty, f"{len(out_rs)} risk_score values outside [0, 1]"
assert out_cf.empty, f"{len(out_cf)} confidence values outside [0, 1]"

print(f"✓ Assertion 2 passed — all {len(predictions):,} risk_scores and confidences are within [0, 1]")

✓ Assertion 2 passed — all 40,954 risk_scores and confidences are within [0, 1]


In [8]:
# ── Assertion 3: risk_level distribution is not degenerate ──

level_shares = predictions['risk_level'].value_counts(normalize=True)
max_share    = level_shares.max()
dominant     = level_shares.idxmax()

if max_share == 1.0:
    print(f"Degenerate: 100% of predictions are '{dominant}'")

assert max_share < 1.0, (
    f"Degenerate distribution: 100% of {len(predictions):,} predictions are '{dominant}'. "
    f"Check that the correct class index was used for risk_score."
)
assert set(predictions['risk_level'].unique()) <= {'high', 'medium', 'low'}, \
    f"Unexpected risk_level values: {set(predictions['risk_level'].unique()) - {'high','medium','low'}}"

print(f"✓ Assertion 3 passed — distribution is non-degenerate")
print(f"  Dominant level '{dominant}' holds {max_share:.1%} of predictions")

✓ Assertion 3 passed — distribution is non-degenerate
  Dominant level 'low' holds 71.6% of predictions


In [9]:
# ── Validation summary — printed only after all three assertions pass ──

rs = predictions['risk_score']
cf = predictions['confidence']

print("=" * 52)
print("VALIDATION SUMMARY")
print("=" * 52)
print(f"Total elevators    :  {len(predictions):,}")
print(f"Columns            :  {len(predictions.columns)} ({len(prob_cols)} prob_* + 7 fixed)")
print()
for level in ('high', 'medium', 'low'):
    n   = (predictions['risk_level'] == level).sum()
    pct = n / len(predictions) * 100
    print(f"  {level:6s}  :  {n:6,}  ({pct:.1f}%)")
print()
print(f"risk_score  min  :  {rs.min():.4f}")
print(f"risk_score  max  :  {rs.max():.4f}")
print(f"risk_score  mean :  {rs.mean():.4f}")
print(f"confidence  min  :  {cf.min():.4f}")
print(f"confidence  max  :  {cf.max():.4f}")
print(f"confidence  mean :  {cf.mean():.4f}")
print()
print("All 3 assertions passed — proceeding to save.")

VALIDATION SUMMARY
Total elevators    :  40,954
Columns            :  20 (13 prob_* + 7 fixed)

  high    :     211  (0.5%)


  medium  :  11,408  (27.9%)
  low     :  29,335  (71.6%)

risk_score  min  :  0.0000
risk_score  max  :  0.9080
risk_score  mean :  0.3301
confidence  min  :  0.1899
confidence  max  :  0.9962
confidence  mean :  0.4437

All 3 assertions passed — proceeding to save.


In [10]:
# ── Save to disk — only reached if all assertions above passed ──

COL_ORDER = (
    ['elevator_id', 'predicted_outcome', 'confidence', 'risk_score', 'risk_level',
     'model_version', 'prediction_date']
    + sorted(prob_cols)  # alphabetical order for prob_* columns
)
out_path = '../data/predictions.csv'

predictions[COL_ORDER].to_csv(out_path, index=False)
print(f"Saved {len(predictions):,} rows × {len(COL_ORDER)} columns → {out_path}")

# Read back and verify.
saved = pd.read_csv(out_path)
assert list(saved.columns) == COL_ORDER, \
    f"Column order mismatch: {list(saved.columns)}"
assert len(saved) == len(predictions), \
    f"Row count mismatch: {len(saved)} vs {len(predictions)}"

print(f"Column order verified ({len(COL_ORDER)} columns).")
print(f"\nElevator 10 row:")
row10 = saved[saved['elevator_id'] == 'EL-00000010']
print(row10[['elevator_id','predicted_outcome','confidence','risk_score','risk_level',
             'model_version','prediction_date']].to_string(index=False))

Saved 40,954 rows × 20 columns → ../data/predictions.csv
Column order verified (20 columns).

Elevator 10 row:
elevator_id   predicted_outcome  confidence  risk_score risk_level model_version prediction_date
EL-00000010 All Orders Resolved      0.6853      0.0296        low          v4.1      2026-06-04
